Render the evaluation images and record the dataset and run hashes.

Pins: primary-bin pins of the gold main sources (14,337) + the 196 clicked edge pins (±1 m, secondary result)
+ primary-bin pins of the gold ablation sources (pilot only, not part of the dataset hash).
Marker `google_pin_red`, same render path as `run_sanity_render`. Images: `data/dataset_v2/renders/<render_id>.png`,
rows appended to the manifest `renders` table (pins already rendered by the sanity render are reused).
`excluded_sources` are rendered too; they are dropped at run/scoring time.

In [1]:
%env DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib

import hashlib
import json
from datetime import UTC, datetime

import duckdb
import geopandas as gpd
import pandas as pd
from rasterio.windows import Window

from thesis.config import REPO_ROOT, load_config
from thesis.data import naip
from thesis.data.manifest import RenderRecord, append_renders
from thesis.render.marker import render_at_tip
from thesis.render.presets import load_presets
from thesis.run import _geometry_to_crs, _save_png, git_commit

out_dir = REPO_ROOT / "data" / "dataset_v2"
manifest_path = out_dir / "manifest.duckdb"
imagery_config = load_config("imagery")
scene_cache = REPO_ROOT / imagery_config["scene_cache_dir"]
PRESET_ID = "google_pin_red"
preset = load_presets(load_config("render"))[PRESET_ID]

env: DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib


In [2]:
selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
boxes = gpd.read_parquet(out_dir / "inventory" / "boxes.parquet")
box_geometries = boxes.set_index("box_id").geometry.to_dict()

gold = selected.merge(pd.read_json(out_dir / "gold" / "is_building.jsonl", lines=True), on="source_id")
primary_bins = load_config("scoring")["primary_bins"]
main_ids = gold.loc[gold["is_building"] & (gold["assignment"] == "main"), "source_id"]
ablation_ids = gold.loc[gold["is_building"] & (gold["assignment"] == "ablation"), "source_id"]
primary = pins.loc[pins["source_id"].isin(main_ids) & pins["bin_id"].isin(primary_bins)]
pilot = pins.loc[pins["source_id"].isin(ablation_ids) & pins["bin_id"].isin(primary_bins)]
edge_marks = pd.read_json(out_dir / "gold" / "edge_marks_phillip.jsonl", lines=True)
edge = pins.loc[pins["pin_id"].isin(edge_marks.loc[~edge_marks["skipped"], "pin_id"])]
to_render = pd.concat(
    [primary.assign(purpose="primary"), edge.assign(purpose="edge"), pilot.assign(purpose="pilot")]
).merge(selected[["source_id", "box_id", "naip_item_id", "scene_window"]], on="source_id")

with duckdb.connect(str(manifest_path), read_only=True) as connection:
    run_id = connection.execute("SELECT run_id FROM runs").fetchone()[0]
    existing = connection.execute("SELECT render_id FROM renders").fetch_df()["render_id"]
to_render["render_id"] = [
    hashlib.sha256(f"{pin_id}:{PRESET_ID}:unchanged".encode()).hexdigest() for pin_id in to_render["pin_id"]
]
new = to_render.loc[~to_render["render_id"].isin(existing)]
to_render["purpose"].value_counts().to_dict(), len(new)

({'primary': 14337, 'pilot': 713, 'edge': 196}, 13779)

In [3]:
# render missing images; scene tiles come from the scene cache (filled during the hand check)
item_records: dict[str, dict[str, dict]] = {}


def scene_tile(source):
    scene_window = Window(*[int(v) for v in source["scene_window"]])
    cache_key = naip.window_cache_key(str(source["naip_item_id"]), scene_window)
    tile = naip.load_cached(scene_cache, cache_key)
    if tile is None:
        box_id = str(source["box_id"])
        if box_id not in item_records:
            box_wgs84 = _geometry_to_crs(box_geometries[box_id], "EPSG:5070", "EPSG:4326")
            item_records[box_id] = {
                str(item["id"]): item for item in naip.search_items_for_geometry(box_wgs84, imagery_config)
            }
        item = item_records[box_id][str(source["naip_item_id"])]
        tile = naip.read_item_window(
            item, scene_window, float(imagery_config["gsd_m"]), float(imagery_config["gsd_tolerance"])
        )
        naip.store_atomic(scene_cache, cache_key, tile.array, tile.metadata)
    return tile


size = int(imagery_config["image_px"])
records = []
for _, source_pins in new.groupby("source_id"):
    tile = scene_tile(source_pins.iloc[0])
    for _, pin in source_pins.iterrows():
        col_off, row_off = int(pin["crop_window"][0]), int(pin["crop_window"][1])
        crop = tile.array[:, row_off : row_off + size, col_off : col_off + size]
        assert crop.shape == (3, size, size), pin["pin_id"]
        relative_path = f"renders/{pin['render_id']}.png"
        records.append(
            RenderRecord(
                run_id=str(run_id),
                render_id=pin["render_id"],
                pin_id=str(pin["pin_id"]),
                preset_id=PRESET_ID,
                marker_shape=str(preset["shape"]),
                marker_color=str(preset["color_name"]),
                marker_size_px=int(preset["size_px"]),
                marker_opacity=float(preset["opacity"]),
                prompt_variant="unchanged",
                image_path=relative_path,
                image_sha256=_save_png(render_at_tip(crop, preset), out_dir / relative_path),
            )
        )
append_renders(manifest_path, records)
len(records)

/Users/phillip/repositories/bachelor-thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/Users/phillip/repositories/bachelor-thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/Users/phillip/repositories/bachelor-thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/Users/phillip/repositories/bachelor-thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/Users/phillip/repositories/bachelor-thesis/.venv/lib/python3.12/site-pa

13779

In [5]:
# dataset hash: every evaluation image; run hash: dataset hash + all post-construction configs
with duckdb.connect(str(manifest_path), read_only=True) as connection:
    renders = connection.execute("SELECT pin_id, image_path, image_sha256 FROM renders").fetch_df()
evaluation = to_render.loc[to_render["purpose"] != "pilot", ["pin_id", "purpose"]].merge(renders, on="pin_id")
assert len(evaluation) == (to_render["purpose"] != "pilot").sum()

dataset_hash = hashlib.sha256(
    "\n".join(sorted(evaluation["pin_id"] + ":" + evaluation["image_sha256"])).encode()
).hexdigest()
configs = {key: load_config(key) for key in ("prompts", "models", "inference", "scoring", "subsamples")}
run_hash = hashlib.sha256(json.dumps({"dataset_hash": dataset_hash, **configs}, sort_keys=True).encode()).hexdigest()
hashes = {
    "dataset_hash": dataset_hash,
    "run_hash": run_hash,
    "images": evaluation["purpose"].value_counts().to_dict(),
    "preset": PRESET_ID,
    "git_commit": git_commit(),
    "created": datetime.now(UTC).isoformat(),
}
(out_dir / "renders" / "hashes.json").write_text(json.dumps(hashes, indent=2))
hashes

{'dataset_hash': 'e8a40dfbf42ad72b1accf16225f8cdc4cbdeefa2ab01e02b0d3fd2ded69836d9',
 'run_hash': 'ebdf4b5e68da0975db2d21d5822b28a9dc400e20a53bfa90209abb7fd662d24f',
 'images': {'primary': 14337, 'edge': 196},
 'preset': 'google_pin_red',
 'git_commit': '85b4e5562f2ea45e3b19cec71dbf48479069f599',
 'created': '2026-10-06T09:37:45.172200+00:00'}